# Notebook 8 — Vérification de la correspondance SNOMED de LSAS (CPU, Internet)

**Réglages :** *Accelerator* **None**, *Internet* **On**. *Inputs* : le dataset du code, la sortie du
notebook 1 (cache, qui contient `lsas/meta.csv`) et la sortie du notebook 4 (runs).

Le script lit les codes SNOMED des 45 152 en-têtes LSAS sur PhysioNet (10 à 15 min), compare notre
correspondance à la liste officielle, puis recalcule les AUROC LSAS avec la correspondance corrigée à partir
des prédictions déjà sauvegardées (aucun réentraînement). Résultats dans `/kaggle/working/lsas_check/`.

In [ ]:
import glob, os, re, shutil
def _version(init):
    m = re.search(r'__version__\s*=\s*"([^"]+)"', open(init).read())
    return m.group(1) if m else "0"
# plusieurs copies du code peuvent être présentes (dataset du code, sorties d'anciens notebooks) :
# on prend la plus récente
hits = sorted(glob.glob("/kaggle/input/**/ecgfusion/__init__.py", recursive=True), key=_version)
assert hits, "Code introuvable : ajoutez le dataset contenant ecg_fusion.zip via « Add Input »."
src = os.path.dirname(os.path.dirname(hits[-1]))
shutil.rmtree("/tmp/ecg_fusion", ignore_errors=True)
shutil.copytree(src, "/tmp/ecg_fusion")   # hors /kaggle/working : le code n'encombre pas la sortie
os.chdir("/tmp/ecg_fusion")
!pip install -q wfdb
print(f"Code version {_version(hits[-1])} prêt (source : {src})")

In [ ]:
runs = []
for n in ["L2_final_s42", "L2_final_s1", "L2_final_s2"]:
    hits = sorted(glob.glob(f"/kaggle/input/**/runs/{n}/eval/summary.csv", recursive=True))
    if hits:
        runs.append(os.path.dirname(os.path.dirname(hits[0])))
metas = sorted(glob.glob("/kaggle/input/**/lsas/meta.csv", recursive=True))
assert metas, "lsas/meta.csv introuvable : ajoutez la sortie du notebook 1 en Input."
print("Runs :", runs); print("Cache LSAS :", os.path.dirname(metas[0]))
prev = sorted(glob.glob("/kaggle/working/lsas_check/lsas_codes.csv") + glob.glob("/kaggle/input/**/lsas_codes.csv", recursive=True))
extra = f"--codes {prev[0]}" if prev else ""   # codes déjà extraits : pas de nouveau téléchargement
print("Codes :", prev[0] if prev else "téléchargement depuis PhysioNet")
!python scripts/lsas_code_check.py --lsas_cache {os.path.dirname(metas[0])} --runs {' '.join(runs)} --out /kaggle/working/lsas_check {extra}